# Chapter 29 Coding Exam — Training Dynamics, Optimizers, Regularization & Early Stopping (100 Points)

**Book Reference:** Chapter 29, Section 29.7 (Coding Exam)  
**Suggested Time:** 45–55 minutes (Open Book, Closed Solution)

| Task | Deep Learning Stabilization & Evaluation Component | Points |
| :--- | :--- | :---: |
| **Task 29.1** | `compare_optimizers_benchmark(...)` (Cloned-init SGD vs. Momentum vs. AdamW table) | 30 |
| **Task 29.2** | `audit_weight_decay_regularization(...)` (AdamW weight-decay sweep & L2 norm audit) | 35 |
| **Task 29.3** | `train_with_early_stopping_mch(...)` (Leak-free 60/20/20 MCH MLP + Cosine LR + Early Stopping) | 35 |
| **TOTAL** | | **100** |

In [ ]:
# @title Step 0: Click Play (▶) to Activate Chapter 29 Exam Grader
!pip install -q sharukh-tutor
import copy
import math
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import sharukh_tutor as tutor

torch.set_num_threads(1)
tutor.activate(chapter="ch29", notebook="coding_exam")


---
## Exam Task 29.1 (30 Points): `compare_optimizers_benchmark(X_train, y_train, X_val, y_val, epochs=35, seed=42)`

Run a fair, controlled optimizer benchmark from **identical initial weights**:
1. Seed with `torch.manual_seed(seed)`, construct a 2-layer MLP `nn.Sequential(nn.Linear(in_dim, 32), nn.ReLU(), nn.Linear(32, 1))`, and deep-copy `base_state = copy.deepcopy(base.state_dict())`.
2. Clone `base_state` into a fresh copy of the MLP for each of:
   - `"SGD"`: `torch.optim.SGD(model.parameters(), lr=0.05)`
   - `"SGD+Momentum"`: `torch.optim.SGD(model.parameters(), lr=0.05, momentum=0.9)`
   - `"AdamW"`: `torch.optim.AdamW(model.parameters(), lr=0.01, weight_decay=1e-2)`
3. Train each for `epochs` full-batch steps with `nn.BCEWithLogitsLoss()`, evaluate in `model.eval()` under `torch.no_grad()`, and return a `pd.DataFrame` with exact columns `["optimizer", "final_train_loss", "final_val_loss", "val_auc"]` rounded to **4 decimals**.

In [ ]:
def compare_optimizers_benchmark(
    X_train, y_train, X_val, y_val, epochs=35, seed=42
):
    """Benchmark SGD, SGD+Momentum & AdamW from cloned initial weights."""
    # STUB: raise NotImplementedError("Replace with your solution")
    ### BEGIN SOLUTION
    Xtr = torch.tensor(np.asarray(X_train, dtype=np.float32))
    ytr = torch.tensor(np.asarray(y_train, dtype=np.float32)).unsqueeze(1)
    Xva = torch.tensor(np.asarray(X_val, dtype=np.float32))
    yva = torch.tensor(np.asarray(y_val, dtype=np.float32)).unsqueeze(1)
    yva_np = np.asarray(y_val, dtype=int)
    in_dim = int(Xtr.shape[1])

    torch.manual_seed(seed)
    base = nn.Sequential(
        nn.Linear(in_dim, 32), nn.ReLU(), nn.Linear(32, 1)
    )
    base_sd = copy.deepcopy(base.state_dict())
    crit = nn.BCEWithLogitsLoss()

    specs = [
        ("SGD", lambda p: torch.optim.SGD(p, lr=0.05)),
        (
            "SGD+Momentum",
            lambda p: torch.optim.SGD(p, lr=0.05, momentum=0.9),
        ),
        (
            "AdamW",
            lambda p: torch.optim.AdamW(p, lr=0.01, weight_decay=1e-2),
        ),
    ]
    rows = []
    for name, make_opt in specs:
        model = nn.Sequential(
            nn.Linear(in_dim, 32), nn.ReLU(), nn.Linear(32, 1)
        )
        model.load_state_dict(copy.deepcopy(base_sd))
        opt = make_opt(model.parameters())
        for _ in range(epochs):
            model.train()
            opt.zero_grad()
            crit(model(Xtr), ytr).backward()
            opt.step()
        model.eval()
        with torch.no_grad():
            tr_l = float(crit(model(Xtr), ytr).item())
            va_l = float(crit(model(Xva), yva).item())
            va_p = torch.sigmoid(model(Xva)).squeeze(1).numpy()
            auc = float(roc_auc_score(yva_np, va_p))
        rows.append({
            "optimizer": name,
            "final_train_loss": round(tr_l, 4),
            "final_val_loss": round(va_l, 4),
            "val_auc": round(auc, 4),
        })
    cols = ["optimizer", "final_train_loss", "final_val_loss", "val_auc"]
    return pd.DataFrame(rows, columns=cols)
    ### END SOLUTION


---
## Exam Task 29.2 (35 Points): `audit_weight_decay_regularization(X_train, y_train, X_val, y_val, weight_decays, epochs=40, seed=42)`

Audit how decoupled weight decay in `AdamW` shrinks weight magnitudes:
1. Seed with `torch.manual_seed(seed)`, build `nn.Sequential(nn.Linear(in_dim, 32), nn.ReLU(), nn.Linear(32, 1))`, and clone `base_sd = copy.deepcopy(base.state_dict())`.
2. For each `wd` in `weight_decays`, reload `copy.deepcopy(base_sd)`, train for `epochs` steps with `AdamW(model.parameters(), lr=0.01, weight_decay=wd)` and `BCEWithLogitsLoss()`, and compute `val_loss`, `val_auc`, and the L2 norm across all 2D weight matrices (`p.ndim >= 2`).
3. Build `table` as a `pd.DataFrame` with columns `["weight_decay", "val_loss", "val_auc", "l2_weight_norm"]` rounded to **4 decimals**, pick `best_wd` from the row with lowest `val_loss`, and return `{"best_wd": float, "table": table}`.

In [ ]:
def audit_weight_decay_regularization(
    X_train, y_train, X_val, y_val, weight_decays, epochs=40, seed=42
):
    """Sweep AdamW weight_decay and audit val loss, AUC & L2 weight norm."""
    # STUB: raise NotImplementedError("Replace with your solution")
    ### BEGIN SOLUTION
    Xtr = torch.tensor(np.asarray(X_train, dtype=np.float32))
    ytr = torch.tensor(np.asarray(y_train, dtype=np.float32)).unsqueeze(1)
    Xva = torch.tensor(np.asarray(X_val, dtype=np.float32))
    yva = torch.tensor(np.asarray(y_val, dtype=np.float32)).unsqueeze(1)
    yva_np = np.asarray(y_val, dtype=int)
    in_dim = int(Xtr.shape[1])

    torch.manual_seed(seed)
    base = nn.Sequential(
        nn.Linear(in_dim, 32), nn.ReLU(), nn.Linear(32, 1)
    )
    base_sd = copy.deepcopy(base.state_dict())
    crit = nn.BCEWithLogitsLoss()

    rows = []
    for wd in weight_decays:
        model = nn.Sequential(
            nn.Linear(in_dim, 32), nn.ReLU(), nn.Linear(32, 1)
        )
        model.load_state_dict(copy.deepcopy(base_sd))
        opt = torch.optim.AdamW(
            model.parameters(), lr=0.01, weight_decay=wd
        )
        for _ in range(epochs):
            model.train()
            opt.zero_grad()
            crit(model(Xtr), ytr).backward()
            opt.step()
        model.eval()
        with torch.no_grad():
            vl = float(crit(model(Xva), yva).item())
            vp = torch.sigmoid(model(Xva)).squeeze(1).numpy()
            auc = float(roc_auc_score(yva_np, vp))
            l2 = math.sqrt(sum(
                float(p.detach().pow(2).sum().item())
                for p in model.parameters()
                if p.ndim >= 2
            ))
        rows.append({
            "weight_decay": round(float(wd), 4),
            "val_loss": round(vl, 4),
            "val_auc": round(auc, 4),
            "l2_weight_norm": round(l2, 4),
        })
    cols = ["weight_decay", "val_loss", "val_auc", "l2_weight_norm"]
    table = pd.DataFrame(rows, columns=cols)
    best_wd = float(table.loc[table["val_loss"].idxmin(), "weight_decay"])
    return {"best_wd": best_wd, "table": table}
    ### END SOLUTION


---
## Exam Task 29.3 (35 Points): `train_with_early_stopping_mch(mch_df, patience=8, max_epochs=80, seed=42)`

Build a leak-free, regularized PyTorch anemia pipeline on `mch_survey_sample` (without mutating `mch_df`):
1. Copy `mch_df`, clean `residence` (`str.strip().str.title()`), replace `anc_visits == 999` with `np.nan`, derive `is_urban` and ordinal `wealth_num` (1..5), and drop NaNs across `["maternal_anemia", "maternal_age_years", "anc_visits", "wealth_num", "is_urban"]` (exclude `maternal_hb_g_dl`!).
2. Stratified 60/20/20 train/val/test split (`test_size=0.40`, then `test_size=0.50`, `stratify=...`, `random_state=seed`) with train-only `StandardScaler`.
3. Seed `torch.manual_seed(seed)`, train a `RegularizedClinicalMLP` (`Linear(4, 32) -> BatchNorm1d(32) -> ReLU -> Dropout(0.3) -> Linear(32, 1)`) using `AdamW(lr=0.02, weight_decay=1e-2)`, `CosineAnnealingLR(opt, T_max=max_epochs)`, `BCEWithLogitsLoss()`, and early stopping (`patience`, deep-copying `best_state_dict`).
4. Restore `best_state_dict`, evaluate `test_auc` in `model.eval()` under `torch.no_grad()`, and return `{"epochs_run": int, "best_val_loss": round5, "test_auc": round4, "stopped_early": bool}`.

In [ ]:
def train_with_early_stopping_mch(
    mch_df, patience=8, max_epochs=80, seed=42
):
    """Train leak-free regularized MCH MLP with early stopping."""
    # STUB: raise NotImplementedError("Replace with your solution")
    ### BEGIN SOLUTION
    work = mch_df.copy()
    work["residence"] = (
        work["residence"].astype(str).str.strip().str.title()
    )
    work["anc_visits"] = work["anc_visits"].replace(999, np.nan)
    work["is_urban"] = (work["residence"] == "Urban").astype(int)
    w_map = {
        "Poorest": 1, "Poorer": 2, "Middle": 3,
        "Richer": 4, "Richest": 5,
    }
    work["wealth_num"] = work["wealth_quintile"].map(w_map)
    feat_cols = [
        "maternal_age_years", "anc_visits", "wealth_num", "is_urban"
    ]
    clean = work[["maternal_anemia"] + feat_cols].dropna()
    X = clean[feat_cols].to_numpy(dtype=np.float32)
    y = clean["maternal_anemia"].astype(int).to_numpy()

    Xtr, Xtmp, ytr, ytmp = train_test_split(
        X, y, test_size=0.40, stratify=y, random_state=seed
    )
    Xva, Xte, yva, yte = train_test_split(
        Xtmp, ytmp, test_size=0.50, stratify=ytmp, random_state=seed
    )
    scaler = StandardScaler()
    Xtr = scaler.fit_transform(Xtr)
    Xva = scaler.transform(Xva)
    Xte = scaler.transform(Xte)

    Xtr_t = torch.tensor(Xtr, dtype=torch.float32)
    ytr_t = torch.tensor(ytr, dtype=torch.float32).unsqueeze(1)
    Xva_t = torch.tensor(Xva, dtype=torch.float32)
    yva_t = torch.tensor(yva, dtype=torch.float32).unsqueeze(1)
    Xte_t = torch.tensor(Xte, dtype=torch.float32)

    torch.manual_seed(seed)
    model = nn.Sequential(
        nn.Linear(4, 32),
        nn.BatchNorm1d(32),
        nn.ReLU(),
        nn.Dropout(0.3),
        nn.Linear(32, 1),
    )
    opt = torch.optim.AdamW(model.parameters(), lr=0.02, weight_decay=1e-2)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(
        opt, T_max=max_epochs
    )
    crit = nn.BCEWithLogitsLoss()

    best_loss, counter, stopped_early = float("inf"), 0, False
    best_sd = None
    epochs_run = 0
    for ep in range(1, max_epochs + 1):
        epochs_run = ep
        model.train()
        opt.zero_grad()
        crit(model(Xtr_t), ytr_t).backward()
        opt.step()
        sched.step()

        model.eval()
        with torch.no_grad():
            vloss = float(crit(model(Xva_t), yva_t).item())
        if vloss < best_loss:
            best_loss = vloss
            counter = 0
            best_sd = copy.deepcopy(model.state_dict())
        else:
            counter += 1
            if counter >= patience:
                stopped_early = True
                break

    if best_sd is not None:
        model.load_state_dict(best_sd)
    model.eval()
    with torch.no_grad():
        probs = torch.sigmoid(model(Xte_t)).squeeze(1).numpy()
    return {
        "epochs_run": int(epochs_run),
        "best_val_loss": round(float(best_loss), 5),
        "test_auc": round(float(roc_auc_score(yte, probs)), 4),
        "stopped_early": bool(stopped_early),
    }
    ### END SOLUTION


In [ ]:
exam_submissions = {
    "ch29_exam_01": compare_optimizers_benchmark,
    "ch29_exam_02": audit_weight_decay_regularization,
    "ch29_exam_03": train_with_early_stopping_mch,
}

tutor.check_exam("ch29", exam_submissions)


---
## Chapter 29 Exam Debugging Post-Mortem (Solution Notebook Only)

- **Task 29.1 (Unfair Optimizer Comparison Without Cloning `state_dict`):** Re-initializing a brand-new random network for each optimizer mixes optimizer quality with random weight luck. Always snapshot `copy.deepcopy(base.state_dict())` once and reload it for every optimizer.
- **Task 29.2 (Weight Norm Including 1D Biases vs. 2D Matrices):** Filtering `if p.ndim >= 2` isolates weight matrices so your L2 norm audit cleanly tracks how decoupled `AdamW` weight decay shrinks learned transformation weights monotonically.
- **Task 29.3 (Evaluating Final Epoch Instead of Restoring `best_state_dict`):** Stopping after `patience=8` worsening epochs leaves `model` sitting at an *overfit* epoch! Calling `model.load_state_dict(best_sd)` and `model.eval()` before scoring `X_test` ensures unbiased test evaluation at the true validation optimum.